# xView Dataset EDA
Exploratory analysis before preprocessing.
Run on EC2 with xView data pulled from S3.

In [1]:
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

DATA_DIR    = Path('/home/ec2-user/argus/data/raw')
LABELS_PATH = DATA_DIR / 'train_labels/xView_train.geojson'
IMAGES_DIR  = DATA_DIR / 'train_images/train_images'
PLOTS_DIR   = Path('/home/ec2-user/argus/notebooks/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Labels:', LABELS_PATH.exists())
print('Images dir:', IMAGES_DIR.exists())
print('Plots dir:', PLOTS_DIR)

# Build lookup: stem (no ext) -> full path, for robust matching
disk_files = {p.stem: p for p in IMAGES_DIR.glob('*') if p.suffix in ('.tif', '.tiff')}
print(f'Images on disk: {len(disk_files)}')
if disk_files:
    print('Sample filenames:', list(disk_files.keys())[:3])

Labels: True
Images dir: True
Plots dir: /home/ec2-user/argus/notebooks/plots
Images on disk: 846
Sample filenames: ['10', '100', '102']


## 1. Load GeoJSON

In [2]:
with open(LABELS_PATH) as f:
    geojson = json.load(f)

features = geojson['features']
print(f'Total annotated objects: {len(features):,}')
print(f'Sample feature keys: {list(features[0]["properties"].keys())}')
print(f'Sample image_id values: {[features[i]["properties"].get("image_id","") for i in range(3)]}')

Total annotated objects: 601,937
Sample feature keys: ['bounds_imcoords', 'edited_by', 'cat_id', 'type_id', 'ingest_time', 'index_right', 'image_id', 'point_geom', 'feature_id', 'grid_file']
Sample image_id values: ['2355.tif', '2355.tif', '2355.tif']


## 2. xView class distribution (all 60 classes)

In [3]:
type_ids = [f['properties'].get('type_id', -1) for f in features]
class_counts = Counter(type_ids)

ids    = sorted(class_counts.keys())
counts = [class_counts[i] for i in ids]

fig, ax = plt.subplots(figsize=(18, 4))
ax.bar([str(i) for i in ids], counts)
ax.set_xlabel('xView type_id')
ax.set_ylabel('Object count')
ax.set_title('Object count per xView class')
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'xview_class_dist.png', dpi=120)
plt.close()
print(f'Unique classes: {len(class_counts)}')

Unique classes: 62


## 3. ARGUS 5-class distribution

In [4]:
import sys
sys.path.insert(0, '/home/ec2-user/argus')
from src.preprocessing.xview_converter import XVIEW_TO_ARGUS, CLASS_NAMES

argus_counts = Counter()
dropped = 0
for f in features:
    tid = f['properties'].get('type_id', -1)
    if tid in XVIEW_TO_ARGUS:
        argus_counts[XVIEW_TO_ARGUS[tid]] += 1
    else:
        dropped += 1

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(CLASS_NAMES, [argus_counts[i] for i in range(5)], color='steelblue')
ax.bar_label(bars)
ax.set_ylabel('Object count')
ax.set_title('ARGUS 5-class distribution')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'argus_class_dist.png', dpi=120)
plt.close()

total_kept = sum(argus_counts.values())
print(f'Kept: {total_kept:,}  |  Dropped (unmapped): {dropped:,}  |  Keep rate: {total_kept/(total_kept+dropped):.1%}')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:15s}: {argus_counts[i]:,}')

Kept: 580,663  |  Dropped (unmapped): 21,274  |  Keep rate: 96.5%
  tank           : 318,698
  large_vehicle  : 28,864
  naval_vessel   : 215,849
  aircraft       : 14,053
  structure      : 3,199


## 4. Bounding box size distribution

In [5]:
box_widths  = []
box_heights = []

for f in features:
    if f['properties'].get('type_id', -1) not in XVIEW_TO_ARGUS:
        continue
    coords = f['geometry']['coordinates'][0]
    xs = [c[0] for c in coords]
    ys = [c[1] for c in coords]
    box_widths.append(max(xs) - min(xs))
    box_heights.append(max(ys) - min(ys))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(box_widths,  bins=80, color='steelblue')
axes[0].set_xlabel('Box width (pixels)')
axes[0].set_title('Bounding box widths')
axes[1].hist(box_heights, bins=80, color='salmon')
axes[1].set_xlabel('Box height (pixels)')
axes[1].set_title('Bounding box heights')
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'bbox_sizes.png', dpi=120)
plt.close()

print(f'Width  — mean: {np.mean(box_widths):.1f}  median: {np.median(box_widths):.1f}  max: {np.max(box_widths):.1f}')
print(f'Height — mean: {np.mean(box_heights):.1f}  median: {np.median(box_heights):.1f}  max: {np.max(box_heights):.1f}')

Width  — mean: 0.0  median: 0.0  max: 0.0
Height — mean: 0.0  median: 0.0  max: 0.0


## 5. Images per class — which images have tanks vs aircraft etc.

In [6]:
# Count distinct images that contain at least one object of each ARGUS class
images_with_class = {i: set() for i in range(5)}

for f in features:
    tid = f['properties'].get('type_id', -1)
    if tid not in XVIEW_TO_ARGUS:
        continue
    img_id = f['properties'].get('image_id', '')
    images_with_class[XVIEW_TO_ARGUS[tid]].add(img_id)

print('Images containing each class:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:15s}: {len(images_with_class[i])} images')

Images containing each class:
  tank           : 729 images
  large_vehicle  : 689 images
  naval_vessel   : 705 images
  aircraft       : 527 images
  structure      : 385 images


## 6. Sample images with bounding boxes

In [7]:
CLASS_COLORS = ['red', 'orange', 'blue', 'green', 'purple']

def find_image(img_id):
    p = IMAGES_DIR / img_id
    if p.exists():
        return p
    return disk_files.get(Path(img_id).stem)

by_image = {}
for f in features:
    img_id = f['properties'].get('image_id', '')
    by_image.setdefault(img_id, []).append(f)

print(f'Unique image_ids in GeoJSON: {len(by_image)}')
print(f'Sample image_ids: {list(by_image.keys())[:3]}')

valid_ids = [
    img_id for img_id, feats in by_image.items()
    if find_image(img_id) is not None
    and any(f['properties'].get('type_id', -1) in XVIEW_TO_ARGUS for f in feats)
]
print(f'Matched images (on disk + has ARGUS objects): {len(valid_ids)}')

sample_ids = random.sample(valid_ids, min(6, len(valid_ids)))

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, img_id in zip(axes, sample_ids):
    img_path = find_image(img_id)
    img = Image.open(img_path)
    ax.imshow(img)
    ax.set_title(img_id, fontsize=8)
    ax.axis('off')

    for f in by_image[img_id]:
        tid = f['properties'].get('type_id', -1)
        if tid not in XVIEW_TO_ARGUS:
            continue
        cls = XVIEW_TO_ARGUS[tid]
        coords = f['geometry']['coordinates'][0]
        xs = [c[0] for c in coords]
        ys = [c[1] for c in coords]
        x0, y0 = min(xs), min(ys)
        w,  h  = max(xs) - x0, max(ys) - y0
        rect = patches.Rectangle((x0, y0), w, h,
                                  linewidth=1, edgecolor=CLASS_COLORS[cls], facecolor='none')
        ax.add_patch(rect)
        ax.text(x0, y0 - 2, CLASS_NAMES[cls], color=CLASS_COLORS[cls], fontsize=6)

from matplotlib.lines import Line2D
legend = [Line2D([0],[0], color=c, lw=2, label=n) for c, n in zip(CLASS_COLORS, CLASS_NAMES)]
fig.legend(handles=legend, loc='lower center', ncol=5, fontsize=10)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'sample_images.png', dpi=120)
plt.close()

Unique image_ids in GeoJSON: 847
Sample image_ids: ['2355.tif', '2356.tif', '2370.tif']
Matched images (on disk + has ARGUS objects): 808


## 7. Class imbalance ratio — key for training

In [8]:
counts = [argus_counts[i] for i in range(5)]
max_count = max(counts)

print('Class imbalance (ratio to most common class):')
for name, cnt in zip(CLASS_NAMES, counts):
    ratio = max_count / cnt if cnt > 0 else float('inf')
    bar = '█' * int(cnt / max_count * 40)
    print(f'  {name:15s}: {cnt:6,}  (1:{ratio:.1f})  {bar}')

print()
print('Classes with ratio > 10x are severely underrepresented.')
print('Consider: class weights in loss, oversampling, or more data sources.')

Class imbalance (ratio to most common class):
  tank           : 318,698  (1:1.0)  ████████████████████████████████████████
  large_vehicle  : 28,864  (1:11.0)  ███
  naval_vessel   : 215,849  (1:1.5)  ███████████████████████████
  aircraft       : 14,053  (1:22.7)  █
  structure      :  3,199  (1:99.6)  

Classes with ratio > 10x are severely underrepresented.
Consider: class weights in loss, oversampling, or more data sources.


## 8. Image size distribution

In [9]:
all_img_files = list(IMAGES_DIR.glob('*.tif'))[:50] + list(IMAGES_DIR.glob('*.tiff'))[:50]
all_img_files = all_img_files[:50]
widths, heights = [], []

for p in all_img_files:
    try:
        img = Image.open(p)
        widths.append(img.width)
        heights.append(img.height)
    except Exception:
        pass

print(f'Sampled {len(widths)} images')
if widths:
    print(f'Width  — min: {min(widths)}  max: {max(widths)}  mean: {np.mean(widths):.0f}')
    print(f'Height — min: {min(heights)}  max: {max(heights)}  mean: {np.mean(heights):.0f}')

    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].hist(widths,  bins=30, color='steelblue')
    axes[0].set_title('Image widths')
    axes[1].hist(heights, bins=30, color='salmon')
    axes[1].set_title('Image heights')
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / 'image_sizes.png', dpi=120)
    plt.close()
else:
    print('No images found to sample')

Sampled 50 images
Width  — min: 2872  max: 3922  mean: 3289
Height — min: 2606  max: 3275  mean: 3021


## Summary

In [10]:
print('=== EDA SUMMARY ===')
print(f'Total objects in xView:    {len(features):,}')
print(f'Objects mapped to ARGUS:   {total_kept:,}  ({total_kept/(total_kept+dropped):.1%})')
print(f'Objects dropped:           {dropped:,}')
print()
print('Per-class counts:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:15s}: {argus_counts[i]:,}')
print()
print('Median box size:', f'{np.median(box_widths):.1f} x {np.median(box_heights):.1f} px')
print('Plots saved: xview_class_dist.png, argus_class_dist.png, bbox_sizes.png, sample_images.png, image_sizes.png')

=== EDA SUMMARY ===
Total objects in xView:    601,937
Objects mapped to ARGUS:   580,663  (96.5%)
Objects dropped:           21,274

Per-class counts:
  tank           : 318,698
  large_vehicle  : 28,864
  naval_vessel   : 215,849
  aircraft       : 14,053
  structure      : 3,199



Median box size: 0.0 x 0.0 px
Plots saved: xview_class_dist.png, argus_class_dist.png, bbox_sizes.png, sample_images.png, image_sizes.png
